# Gold KPI Inspection

This notebook inspects the Gold maintenance marts written by the Gold pipeline.

It is intended as a dashboard-seed notebook for Databricks and focuses on:
- current fleet AOG and open-issue status
- daily operations trends
- component pressure and reliability indicators
- a fleet watchlist for immediate operational review

In [1]:
import os

storage_account = os.getenv("JETOPS_STORAGE_ACCOUNT", "stherbalifedev001")
gold_container = os.getenv("JETOPS_GOLD_CONTAINER", "gold")
secret_scope = os.getenv("JETOPS_SECRET_SCOPE", "herbalife-storage")
gold_secret_key = os.getenv("JETOPS_GOLD_SECRET_KEY", "raw-sas-token")
storage_account_key_secret = os.getenv("JETOPS_STORAGE_ACCOUNT_KEY_SECRET", "storage-account-key")
storage_auth_mode = os.getenv("JETOPS_STORAGE_AUTH_MODE", "account_key")
gold_dataset_root = os.getenv("JETOPS_GOLD_DATASET_ROOT", "jetops/maintenance_kpis")
daily_operations_dataset = os.getenv("JETOPS_GOLD_DAILY_DATASET", f"{gold_dataset_root}/daily_operations")
component_reliability_dataset = os.getenv("JETOPS_GOLD_COMPONENT_DATASET", f"{gold_dataset_root}/component_reliability")
fleet_status_dataset = os.getenv("JETOPS_GOLD_FLEET_DATASET", f"{gold_dataset_root}/fleet_status_snapshot")

daily_operations_path = f"wasbs://{gold_container}@{storage_account}.blob.core.windows.net/{daily_operations_dataset}"
component_reliability_path = f"wasbs://{gold_container}@{storage_account}.blob.core.windows.net/{component_reliability_dataset}"
fleet_status_path = f"wasbs://{gold_container}@{storage_account}.blob.core.windows.net/{fleet_status_dataset}"

is_databricks = "dbutils" in globals() and "spark" in globals()
print(f"Execution mode: {'databricks' if is_databricks else 'local'}")
print(f"Daily operations path: {daily_operations_path}")
print(f"Component reliability path: {component_reliability_path}")
print(f"Fleet status path: {fleet_status_path}")

if is_databricks:
    if storage_auth_mode == "sas":
        gold_sas_token = dbutils.secrets.get(scope=secret_scope, key=gold_secret_key)
        spark.conf.set(
            f"fs.azure.sas.{gold_container}.{storage_account}.blob.core.windows.net",
            gold_sas_token,
        )
    else:
        storage_account_key = dbutils.secrets.get(scope=secret_scope, key=storage_account_key_secret)
        spark.conf.set(
            f"fs.azure.account.key.{storage_account}.blob.core.windows.net",
            storage_account_key,
        )
else:
    print("This inspection notebook is primarily intended for Databricks because it reads Gold Delta datasets directly.")

Execution mode: local
Daily operations path: wasbs://gold@stherbalifedev001.blob.core.windows.net/jetops/maintenance_kpis/daily_operations
Component reliability path: wasbs://gold@stherbalifedev001.blob.core.windows.net/jetops/maintenance_kpis/component_reliability
Fleet status path: wasbs://gold@stherbalifedev001.blob.core.windows.net/jetops/maintenance_kpis/fleet_status_snapshot
This inspection notebook is primarily intended for Databricks because it reads Gold Delta datasets directly.


In [2]:
if is_databricks:
    daily_operations_df = spark.read.format("delta").load(daily_operations_path)
    component_reliability_df = spark.read.format("delta").load(component_reliability_path)
    fleet_status_df = spark.read.format("delta").load(fleet_status_path)

    daily_operations_df.createOrReplaceTempView("gold_daily_operations")
    component_reliability_df.createOrReplaceTempView("gold_component_reliability")
    fleet_status_df.createOrReplaceTempView("gold_fleet_status_snapshot")

    print(f"Daily operations rows: {daily_operations_df.count()}")
    print(f"Component reliability rows: {component_reliability_df.count()}")
    print(f"Fleet status snapshot rows: {fleet_status_df.count()}")
else:
    print("Run this notebook in Databricks to query the Gold Delta marts.")

Run this notebook in Databricks to query the Gold Delta marts.


In [ ]:
if is_databricks:
    print("Executive KPI snapshot")
    display(spark.sql("""
        SELECT
            COUNT(*) AS fleet_aircraft,
            SUM(aog_flag) AS aircraft_in_aog,
            SUM(open_issue_flag) AS aircraft_with_open_issues,
            SUM(CASE WHEN current_severity = 'Critical' THEN 1 ELSE 0 END) AS aircraft_with_critical_issues
        FROM gold_fleet_status_snapshot
    """))

    print("Daily operations trend, last 14 days")
    display(spark.sql("""
        SELECT
            event_date,
            SUM(total_events) AS total_events,
            SUM(open_events) AS open_events,
            SUM(aog_events) AS aog_events,
            SUM(critical_events) AS critical_events
        FROM gold_daily_operations
        WHERE event_date >= date_sub(current_date(), 14)
        GROUP BY event_date
        ORDER BY event_date DESC
    """))
else:
    print("Databricks mode required for KPI queries.")

In [ ]:
if is_databricks:
    print("Top component pressure over the last 7 days")
    display(spark.sql("""
        SELECT
            component,
            SUM(total_events) AS total_events_7d,
            SUM(aog_events) AS aog_events_7d,
            SUM(critical_events) AS critical_events_7d,
            SUM(unscheduled_events) AS unscheduled_events_7d,
            ROUND(AVG(avg_inspection_age_hours), 2) AS avg_inspection_age_hours
        FROM gold_component_reliability
        WHERE event_date >= date_sub(current_date(), 7)
        GROUP BY component
        ORDER BY aog_events_7d DESC, critical_events_7d DESC, total_events_7d DESC
    """))

    print("Current AOG concentration by station")
    display(spark.sql("""
        SELECT
            airport_code,
            hangar,
            COUNT(*) AS aircraft_in_aog
        FROM gold_fleet_status_snapshot
        WHERE aog_flag = 1
        GROUP BY airport_code, hangar
        ORDER BY aircraft_in_aog DESC, airport_code, hangar
    """))
else:
    print("Databricks mode required for Gold mart aggregations.")

In [ ]:
if is_databricks:
    print("Fleet watchlist")
    display(spark.sql("""
        SELECT
            tail_number,
            aircraft_model,
            airport_code,
            hangar,
            current_status,
            current_severity,
            current_component,
            current_maintenance_type,
            current_fault_code,
            current_technician_id,
            days_since_latest_event,
            latest_event_timestamp
        FROM gold_fleet_status_snapshot
        WHERE aog_flag = 1 OR open_issue_flag = 1 OR current_severity IN ('High', 'Critical')
        ORDER BY aog_flag DESC, current_severity DESC, latest_event_timestamp DESC
    """))
else:
    print("Databricks mode required for the fleet watchlist view.")